# Notebook 8 — Model Export & FastAPI Integration

**Goal:** Package the full inference pipeline into a single `.joblib` file
and verify it works end-to-end before deploying via FastAPI.

**Pipeline contents:**
1. Per-dataset MinMaxScaler
2. Savitzky-Golay noise smoother
3. Linear imputer for missing values
4. Time-window constructor
5. Trained LSTM model (weights loaded at prediction time)
6. CUSUM health-state classifier


In [ ]:
import sys
sys.path.append('../')

import numpy as np
import pandas as pd
import joblib
import os
import json
import requests
import tensorflow as tf
from sklearn.base import BaseEstimator, RegressorMixin

from src.data_loader    import load_all_datasets, FEATURE_COLS, SENSOR_COLS
from src.preprocessor   import (apply_savgol_filter, impute_missing,
                                 full_preprocess_pipeline)
from src.windowing      import create_windows
from src.models.lstm_baseline import build_lstm_baseline
from src.changepoint    import cusum_detector, classify_health_state
from src.explainer      import (build_shap_explainer, compute_shap_values,
                                 aggregate_shap_by_feature, build_explanation_text)

WINDOW_SIZE = 30
MAX_RUL     = 125
os.makedirs('../models/saved', exist_ok=True)


## 8.1 Define the Production Pipeline Wrapper


In [ ]:
class PredictiveMaintenancePipeline(BaseEstimator, RegressorMixin):
    """
    Joblib-serialisable wrapper combining the full inference chain:
        raw sensor array
            → Savitzky-Golay smoothing
            → linear imputation of missing values
            → MinMaxScaler normalisation
            → time-window construction
            → LSTM RUL prediction
            → CUSUM health state classification

    Usage:
        pipeline = PredictiveMaintenancePipeline(...)
        result   = pipeline.predict(X_raw)

        X_raw : np.ndarray of shape (n_cycles, 24)
                columns = op_setting_1-3, sensor_1-21 (in order)

    The LSTM model weights are lazy-loaded on the first predict() call and
    cached in self._model (not serialised by joblib — weights path is stored).
    """

    def __init__(self,
                  scaler,
                  model_weights_path: str,
                  feature_cols: list,
                  window_size: int = 30,
                  max_rul: int = 125,
                  cusum_threshold: float = 5.0,
                  sg_window: int = 11,
                  sg_poly: int = 3):
        self.scaler             = scaler
        self.model_weights_path = model_weights_path
        self.feature_cols       = feature_cols
        self.window_size        = window_size
        self.max_rul            = max_rul
        self.cusum_threshold    = cusum_threshold
        self.sg_window          = sg_window
        self.sg_poly            = sg_poly
        self._model             = None   # not serialised

    def _load_model(self):
        """Lazy-load LSTM weights on first call."""
        if self._model is None:
            self._model = build_lstm_baseline(
                window_size=self.window_size,
                n_features=len(self.feature_cols)
            )
            self._model.load_weights(self.model_weights_path)

    def predict(self, X_raw: np.ndarray) -> dict:
        """
        Full inference pipeline.

        Args:
            X_raw : np.ndarray of shape (n_cycles, 24) — raw sensor readings
                    ordered by cycle (oldest → most recent)

        Returns:
            dict with:
                rul_prediction        : float (cycles remaining)
                health_state          : str ('Healthy' | 'Warning' | 'Critical')
                change_point_detected : bool
                change_point_step     : int or None
        """
        from scipy.signal import savgol_filter

        self._load_model()
        X = X_raw.astype(np.float64).copy()

        # 1. Smooth noise (per column, like per-unit smoothing in training)
        if len(X) >= self.sg_window:
            for j in range(X.shape[1]):
                X[:, j] = savgol_filter(X[:, j], self.sg_window, self.sg_poly)

        # 2. Impute missing values (linear fill within array)
        X = pd.DataFrame(X, columns=self.feature_cols)
        X = X.interpolate(method='linear', limit_direction='both').values

        # 3. Normalise
        X_norm = self.scaler.transform(X)

        # 4. Build last window
        T = len(X_norm)
        if T < self.window_size:
            pad    = np.zeros((self.window_size - T, X_norm.shape[1]))
            X_norm = np.vstack([pad, X_norm])
        window = X_norm[-self.window_size:][np.newaxis, :, :]  # (1, Tw, F)

        # 5. Predict RUL
        rul = float(self._model.predict(window, verbose=0).flatten()[0])
        rul = float(np.clip(rul, 0, self.max_rul))

        # 6. Change-point detection on last 50 cycles
        n_recent = min(50, len(X_norm))
        recent   = X_norm[-n_recent:]
        # Monitor the most informative sensor (sensor_11 at index 13)
        sensor_11_idx = self.feature_cols.index('sensor_11')
        cp = cusum_detector(recent[:, sensor_11_idx],
                             threshold=self.cusum_threshold)

        health = classify_health_state(
            rul_prediction=rul,
            change_point_detected=(cp is not None),
            critical_rul=20.0, warning_rul=50.0
        )

        return {
            'rul_prediction':        round(rul, 1),
            'health_state':          health,
            'change_point_detected': cp is not None,
            'change_point_step':     int(cp) if cp is not None else None
        }

    def __getstate__(self):
        """Exclude the loaded model from joblib serialisation."""
        state = self.__dict__.copy()
        state['_model'] = None
        return state

    def __setstate__(self, state):
        """Restore state; model will be lazy-loaded on next predict()."""
        self.__dict__.update(state)
        self._model = None


## 8.2 Build and Save Pipelines for All Datasets


In [ ]:
datasets = load_all_datasets(data_dir='../data/raw')

for ds_id in ['FD001', 'FD002', 'FD003', 'FD004']:
    print(f"\nBuilding pipeline for {ds_id}...")

    df_tr, df_te, scaler = full_preprocess_pipeline(
        df_train=datasets[ds_id]['train'],
        df_test=datasets[ds_id]['test'],
        feature_cols=FEATURE_COLS, sensor_cols=SENSOR_COLS,
        smooth=True, max_rul=MAX_RUL,
        scaler_save_path=f'../models/saved/scaler_{ds_id}.joblib'
    )
    datasets[ds_id]['train_norm'] = df_tr

    # Check if trained weights exist (run NB04 first if not)
    weights_path = f'../models/saved/lstm_target_only_{ds_id}.keras'
    if not os.path.exists(weights_path):
        print(f"  Weights not found — training baseline model for {ds_id}...")
        from tensorflow.keras.callbacks import EarlyStopping
        from sklearn.model_selection import train_test_split

        X, y, _ = create_windows(df_tr, FEATURE_COLS, WINDOW_SIZE)
        X_tr, X_val, y_tr, y_val = train_test_split(
            X.astype(np.float32), y.astype(np.float32),
            test_size=0.1, random_state=42
        )
        m = build_lstm_baseline(WINDOW_SIZE, len(FEATURE_COLS))
        m.fit(X_tr, y_tr, validation_data=(X_val, y_val),
               epochs=100, batch_size=256,
               callbacks=[EarlyStopping(patience=20, restore_best_weights=True)],
               verbose=0)
        m.save_weights(weights_path)
        print(f"  Weights saved: {weights_path}")

    pipeline = PredictiveMaintenancePipeline(
        scaler=scaler,
        model_weights_path=weights_path,
        feature_cols=FEATURE_COLS,
        window_size=WINDOW_SIZE,
        max_rul=MAX_RUL,
        cusum_threshold=5.0
    )

    pipeline_path = f'../models/saved/pm_pipeline_{ds_id.lower()}.joblib'
    joblib.dump(pipeline, pipeline_path)
    print(f"  Pipeline saved: {pipeline_path}")

print("\nAll pipelines exported.")


## 8.3 Verify Pipeline End-to-End


In [ ]:
# Load the FD001 pipeline and run a test prediction
pipeline = joblib.load('../models/saved/pm_pipeline_fd001.joblib')

# Use a real engine from the test set
df_test = datasets['FD001']['test']
unit_1_raw = df_test[df_test['unit_id'] == 1][FEATURE_COLS].values

result = pipeline.predict(unit_1_raw)
print("Pipeline test prediction:")
for k, v in result.items():
    print(f"  {k}: {v}")


In [ ]:
# Verify prediction changes with engine age
df_train_fd001 = datasets['FD001']['train']
unit_max_cycle = df_train_fd001[df_train_fd001['unit_id'] == 1]['cycle'].max()

rul_trajectory = []
for pct in [0.2, 0.4, 0.6, 0.8, 1.0]:
    n_cycles = int(unit_max_cycle * pct)
    raw_data = df_train_fd001[
        (df_train_fd001['unit_id'] == 1) &
        (df_train_fd001['cycle'] <= n_cycles)
    ][FEATURE_COLS].values
    r = pipeline.predict(raw_data)
    rul_trajectory.append({
        'Pct Lifetime': f'{pct*100:.0f}%',
        'Cycles Used':  n_cycles,
        'Predicted RUL': r['rul_prediction'],
        'Health State':  r['health_state'],
        'Change Point':  r['change_point_detected']
    })

traj_df = pd.DataFrame(rul_trajectory)
print("\nPipeline predictions at different lifecycle stages:")
print(traj_df.to_string(index=False))


In [ ]:
# Visualise predicted RUL over lifecycle stages
fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(traj_df['Cycles Used'], traj_df['Predicted RUL'],
        marker='o', linewidth=2.5, color='steelblue', markersize=8)
ax.axhline(50, color='orange', linestyle='--', linewidth=1.5, label='Warning threshold (50)')
ax.axhline(20, color='red',    linestyle='--', linewidth=1.5, label='Critical threshold (20)')

for _, row in traj_df.iterrows():
    color = ('red' if row['Health State'] == 'Critical'
             else ('orange' if row['Health State'] == 'Warning' else 'green'))
    ax.annotate(row['Health State'],
                xy=(row['Cycles Used'], row['Predicted RUL']),
                xytext=(0, 12), textcoords='offset points',
                ha='center', color=color, fontsize=9)

ax.set_xlabel('Cycles of Data Available')
ax.set_ylabel('Predicted RUL')
ax.set_title('Pipeline Predictions at Different Lifecycle Stages (Engine 1, FD001)')
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()


**Insight:** The predicted RUL decreases monotonically as more cycles become
available and the engine approaches failure. The health state transitions from
Healthy → Warning → Critical, providing a natural three-stage alert system
for factory operators.


## 8.4 FastAPI Startup Check


In [ ]:
# Verify API can start by checking import paths
print("FastAPI module check:")
try:
    from api.schemas   import PredictRequest, PredictResponse
    from api.predictor import run_prediction, list_available_models
    from api.main      import app
    print("  All API modules imported successfully.")
    models_available = list_available_models(models_dir='../models/saved')
    print(f"  Available models: {models_available}")
except ImportError as e:
    print(f"  Import error: {e}")
    print("  Ensure api/__init__.py exists and all api/ files are created.")


## 8.5 API Startup & Test Instructions


In [ ]:
startup_instructions = """
To launch the FastAPI server, run from the project root directory:

    uvicorn api.main:app --host 0.0.0.0 --port 8000 --reload

Then open your browser to:
    http://localhost:8000/docs   ← Swagger UI (interactive API explorer)
    http://localhost:8000/redoc  ← ReDoc documentation

Available endpoints:
    GET  /health   → liveness check
    GET  /models   → list available dataset models
    POST /predict  → RUL prediction + health state + explanation

Example Python request (run after starting the server):

    import requests, json
    payload = {
        "unit_id":    "engine_001",
        "dataset_id": "FD001",
        "readings":   raw_sensor_data.tolist()   # shape: (n_cycles, 24)
    }
    r = requests.post("http://localhost:8000/predict", json=payload)
    print(json.dumps(r.json(), indent=2))
"""
print(startup_instructions)


In [ ]:
# Live API test (run this cell AFTER starting the server in a separate terminal)
try:
    # Test /health
    r = requests.get("http://localhost:8000/health", timeout=3)
    print("Health check:", r.json())

    # Test /models
    r = requests.get("http://localhost:8000/models", timeout=3)
    print("Available models:", r.json())

    # Test /predict
    unit_1_readings = df_test[df_test['unit_id'] == 1][FEATURE_COLS].values
    payload = {
        "unit_id":    "engine_001",
        "dataset_id": "FD001",
        "readings":   unit_1_readings.tolist()
    }
    r = requests.post("http://localhost:8000/predict", json=payload, timeout=10)
    print("\nPrediction response:")
    print(json.dumps(r.json(), indent=2))

except requests.exceptions.ConnectionError:
    print("Server not running. Start it with: uvicorn api.main:app --port 8000")


## 8.6 Final Export Summary


In [ ]:
print("=" * 55)
print("EXPORTED FILES IN models/saved/")
print("=" * 55)
for f in sorted(os.listdir('../models/saved')):
    size_kb = os.path.getsize(f'../models/saved/{f}') / 1024
    print(f"  {f:<45} {size_kb:>8.1f} KB")

print("\n" + "=" * 55)
print("DEPLOYMENT CHECKLIST")
print("=" * 55)
checklist = [
    "pm_pipeline_fd001.joblib  — FD001 full inference pipeline",
    "pm_pipeline_fd002.joblib  — FD002 full inference pipeline",
    "pm_pipeline_fd003.joblib  — FD003 full inference pipeline",
    "pm_pipeline_fd004.joblib  — FD004 full inference pipeline",
    "lstm_target_only_*.keras  — LSTM weights per dataset",
    "scaler_*.joblib           — MinMaxScaler per dataset",
]
for item in checklist:
    path_key = item.split('—')[0].strip().replace('*', 'FD001')
    exists = os.path.exists(f'../models/saved/{path_key}')
    status = "✓" if exists else "✗ (MISSING)"
    print(f"  [{status}] {item}")

print("\nTo add a NEW machine type:")
print("  1. Collect run-to-failure sensor data in (n_cycles × 24) format")
print("  2. Run full_preprocess_pipeline() → save scaler as scaler_NEWTYPE.joblib")
print("  3. Train LSTM → save weights as lstm_target_only_NEWTYPE.keras")
print("  4. Create PredictiveMaintenancePipeline → save as pm_pipeline_newtype.joblib")
print("  5. Call POST /predict with dataset_id='NEWTYPE'")
